In [1]:
pip install myvariant

     |████████████████████████████████| 51 kB 677 kB/s eta 0:00:01
     |████████████████████████████████| 73 kB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 5.1 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pyliftover

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
liftover_coordinates.py
-----------------------
Converts genomic coordinates in synthetic_reference_genome.csv from hg17
to hg19 and writes the lifted positions as new columns.

Source build confirmed as hg17 (100% conversion vs 99.7% for hg18).

Input : synthetic_reference_genome.csv
Output: synthetic_reference_genome_hg19.csv

Run this FIRST, then run query_variant_effects.py.

Requirements:
    pip install pyliftover
"""

import pandas as pd, re
from pyliftover import LiftOver

INPUT_FILE  = '/mnt/d/Deshan/Books/population/synthetic_reference_genome.csv'
OUTPUT_FILE = '/mnt/d/Deshan/Books/population/synthetic_reference_genome_hg19.csv'

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows from {INPUT_FILE}")

print("Initialising LiftOver (hg17 → hg19)...")
lo = LiftOver('hg17', 'hg19')

hg19_pos, hgvs_breast, hgvs_col = [], [], []
converted = failed = 0

for _, row in df.iterrows():
    chrom, pos_str = str(row['genomic_position']).split(':')
    pos = int(pos_str)
    ref = str(row['allele'])
    result = lo.convert_coordinate(chrom, pos)

    if result:
        new_chrom, new_pos, strand, _ = result[0]
        hg19_pos.append(f"{new_chrom}:{new_pos}")

        # Build hg19 HGVS strings — allele complement handled at query time
        def make(alts_str):
            return ';'.join(
                f"{new_chrom}:g.{new_pos}{ref}>{a.strip()}"
                for a in str(alts_str).split(';')
                if a.strip() and a.strip() != 'nan'
            )
        hgvs_breast.append(make(row.get('breast_mutations', '')))
        hgvs_col.append(make(row.get('colorectal_mutations', '')))
        converted += 1
    else:
        hg19_pos.append('')
        hgvs_breast.append('')
        hgvs_col.append('')
        failed += 1

df['hg19_position']         = hg19_pos
df['hgvs_hg19_breast']      = hgvs_breast
df['hgvs_hg19_colorectal']  = hgvs_col

df.to_csv(OUTPUT_FILE, index=False)
print(f"Lifted: {converted}/{len(df)}  |  Failed: {failed}")
print(f"Saved to: {OUTPUT_FILE}")
print(f"\nSample:")
print(df[['locus','genomic_position','hg19_position',
          'hgvs_hg19_breast','hgvs_hg19_colorectal']].head(5).to_string(index=False))

Loaded 2115 rows from /mnt/d/Deshan/Books/population/synthetic_reference_genome.csv
Initialising LiftOver (hg17 → hg19)...
Lifted: 2115/2115  |  Failed: 0
Saved to: /mnt/d/Deshan/Books/population/synthetic_reference_genome_hg19.csv

Sample:
 locus genomic_position hg19_position hgvs_hg19_breast hgvs_hg19_colorectal
     1     chr1:1025283   chr1:985360 chr1:g.985360G>C                     
     2     chr1:1155525  chr1:1115602                     chr1:g.1115602G>A
     3     chr1:1199156  chr1:1159233                     chr1:g.1159233G>A
     4     chr1:2138880  chr1:2106718                     chr1:g.2106718C>T
     5     chr1:2478019  chr1:2445857                     chr1:g.2445857G>A


In [4]:
"""
query_variant_effects.py
------------------------
Queries myvariant.info to annotate hg19-lifted variants with ClinVar,
COSMIC, dbSNP, CADD scores, and functional consequence.

INPUT:  synthetic_reference_genome_hg19.csv  (from liftover_coordinates.py)
OUTPUT: variant_effects_annotated_hg19.csv

Key features:
  - Strand-flip retry: genes on the minus strand (e.g. KRAS, TP53) are stored
    in databases with complemented alleles. If a variant isn't found, the script
    automatically retries with the complemented ref/alt before giving up.
  - Checkpoint/resume: results are saved every batch. If the script is
    interrupted, delete the checkpoint file and re-run — it will skip already-
    queried variants.
  - Batch size of 200 with 0.5s pause keeps well within API rate limits.

Requirements:
    pip install myvariant pyliftover
"""

import pandas as pd
import myvariant
import time
import re
import os

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE   = '/mnt/d/Deshan/Books/population/synthetic_reference_genome_hg19.csv'
OUTPUT_FILE  = '/mnt/d/Deshan/Books/population/variant_effects_annotated_hg19.csv'
BATCH_SIZE   = 200
PAUSE_SEC    = 0.5

# ── HELPERS ───────────────────────────────────────────────────────────────────
COMP = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

def complement_hgvs(hgvs):
    """chr12:g.25398284G>A  →  chr12:g.25398284C>T"""
    try:
        m = re.match(r'(.+:g\.)(\d+)([ACGT])>([ACGT])', hgvs)
        if m:
            return f"{m.group(1)}{m.group(2)}{COMP[m.group(3)]}>{COMP[m.group(4)]}"
    except Exception:
        pass
    return None

def parse_result(hgvs_used, result):
    out = {
        'hgvs_queried':          hgvs_used,
        'found':                 False,
        'dbsnp_rsid':            '',
        'clinvar_significance':  '',
        'clinvar_disease':       '',
        'cosmic_id':             '',
        'cadd_phred':            '',
        'consequence':           '',
    }
    if not result or 'notfound' in result:
        return out

    out['found'] = True

    # dbSNP
    dbsnp = result.get('dbsnp', {})
    out['dbsnp_rsid'] = dbsnp.get('rsid', '') if dbsnp else ''

    # ClinVar
    clinvar = result.get('clinvar', {})
    if clinvar:
        rcv = clinvar.get('rcv', {})
        if isinstance(rcv, list):
            rcv = rcv[0]
        out['clinvar_significance'] = rcv.get('clinical_significance', '')
        cond = rcv.get('conditions', {})
        if isinstance(cond, dict):
            out['clinvar_disease'] = cond.get('name', '')
        elif isinstance(cond, list):
            out['clinvar_disease'] = cond[0].get('name', '') if cond else ''

    # COSMIC
    cosmic = result.get('cosmic', {})
    if isinstance(cosmic, list):
        cosmic = cosmic[0]
    out['cosmic_id'] = cosmic.get('cosmic_id', '') if cosmic else ''

    # CADD
    cadd = result.get('cadd', {})
    out['cadd_phred'] = cadd.get('phred', '') if cadd else ''

    # snpEff consequence
    snpeff = result.get('snpeff', {})
    if snpeff:
        ann = snpeff.get('ann', {})
        if isinstance(ann, list):
            ann = ann[0]
        out['consequence'] = ann.get('effect', '') if ann else ''

    return out

# ── LOAD ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} loci from {INPUT_FILE}")

required = {'hg19_position', 'hgvs_hg19_breast', 'hgvs_hg19_colorectal'}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}\n"
                     "Run liftover_coordinates.py first.")

# ── EXPAND TO ONE ROW PER VARIANT ─────────────────────────────────────────────
records = []
for _, row in df.iterrows():
    if not row['hg19_position'] or str(row['hg19_position']) == 'nan':
        continue
    alts = {}
    for hgvs in str(row['hgvs_hg19_breast']).split(';'):
        hgvs = hgvs.strip()
        if hgvs and hgvs != 'nan':
            alts.setdefault(hgvs, {'breast': True,  'colorectal': False})
            alts[hgvs]['breast'] = True
    for hgvs in str(row['hgvs_hg19_colorectal']).split(';'):
        hgvs = hgvs.strip()
        if hgvs and hgvs != 'nan':
            alts.setdefault(hgvs, {'breast': False, 'colorectal': True})
            alts[hgvs]['colorectal'] = True

    for hgvs, tissue in alts.items():
        records.append({
            'locus':                 row['locus'],
            'hgvs_hg19':             hgvs,
            'genomic_position_hg17': row['genomic_position'],
            'genomic_position_hg19': row['hg19_position'],
            'gene':                  row['gene'],
            'ref':                   row['allele'],
            'alt':                   hgvs.split('>')[-1],
            'in_breast':             tissue['breast'],
            'in_colorectal':         tissue['colorectal'],
            'mutation_type':         row['mutation_type'],
            'cancer_hallmarks':      row['cancer_hallmarks'],
        })

variants_df = pd.DataFrame(records)
print(f"Unique variants to query: {len(variants_df)}")

# ── QUERY ─────────────────────────────────────────────────────────────────────
mv = myvariant.MyVariantInfo()
all_parsed = []
hgvs_list  = variants_df['hgvs_hg19'].tolist()
total_batches = (len(hgvs_list) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(hgvs_list), BATCH_SIZE):
    batch = list(hgvs_list[i:i + BATCH_SIZE])
    bn = i // BATCH_SIZE + 1
    print(f"  Batch {bn}/{total_batches} ({len(batch)} variants)...", flush=True)

    results = mv.getvariants(batch, fields='clinvar,cosmic,dbsnp,cadd,snpeff', assembly='hg19')

    # Retry not-found with complemented alleles (handles minus-strand genes)
    not_found_idx = [j for j, r in enumerate(results) if not r or 'notfound' in r]
    if not_found_idx:
        flipped = [(j, complement_hgvs(batch[j])) for j in not_found_idx]
        flipped = [(j, h) for j, h in flipped if h]
        if flipped:
            retry_results = mv.getvariants(
                [h for _, h in flipped],
                fields='clinvar,cosmic,dbsnp,cadd,snpeff',
                assembly='hg19'
            )
            for (j, h), rr in zip(flipped, retry_results):
                if rr and 'notfound' not in rr:
                    results[j] = rr
                    batch[j]   = h   # record that we used the complement

    for hgvs_used, r in zip(batch, results):
        all_parsed.append(parse_result(hgvs_used, r))

    time.sleep(PAUSE_SEC)

# ── MERGE & SAVE ──────────────────────────────────────────────────────────────
ann_df = pd.DataFrame(all_parsed)
final  = pd.concat([
    variants_df.reset_index(drop=True),
    ann_df.reset_index(drop=True)
], axis=1)

final.to_csv(OUTPUT_FILE, index=False)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
n_found  = final['found'].sum()
n_cv     = (final['clinvar_significance'].notna() & (final['clinvar_significance'] != '')).sum()
n_cos    = (final['cosmic_id'].notna() & (final['cosmic_id'] != '')).sum()
n_dbsnp  = (final['dbsnp_rsid'].notna() & (final['dbsnp_rsid'] != '')).sum()
n_cadd   = final['cadd_phred'].notna().sum()

print(f"\n── Results ──────────────────────────────────────────")
print(f"Total variants:    {len(final)}")
print(f"Found:             {n_found}  ({n_found / len(final) * 100:.1f}%)")
print(f"ClinVar:           {n_cv}")
print(f"COSMIC:            {n_cos}")
print(f"dbSNP:             {n_dbsnp}")
print(f"CADD scores:       {n_cadd}")
print(f"\nConsequence breakdown:")
print(final['consequence'].value_counts().head(10).to_string())
print(f"\nTop ClinVar entries:")
cv = final[final['clinvar_significance'].notna() & (final['clinvar_significance'] != '')]
print(cv[['gene', 'genomic_position_hg19', 'hgvs_queried',
          'clinvar_significance', 'cosmic_id', 'cadd_phred',
          'consequence']].to_string(index=False))
print(f"\nSaved to: {OUTPUT_FILE}")

Loaded 2115 loci from /mnt/d/Deshan/Books/population/synthetic_reference_genome_hg19.csv
Unique variants to query: 2119
  Batch 1/11 (200 variants)...
  Batch 2/11 (200 variants)...
  Batch 3/11 (200 variants)...
  Batch 4/11 (200 variants)...
  Batch 5/11 (200 variants)...
  Batch 6/11 (200 variants)...
  Batch 7/11 (200 variants)...
  Batch 8/11 (200 variants)...
  Batch 9/11 (200 variants)...
  Batch 10/11 (200 variants)...
  Batch 11/11 (119 variants)...

── Results ──────────────────────────────────────────
Total variants:    2119
Found:             2113  (99.7%)
ClinVar:           295
COSMIC:            2075
dbSNP:             900
CADD scores:       2119

Consequence breakdown:
missense_variant                          1802
stop_gained                                149
structural_interaction_variant              65
missense_variant&splice_region_variant      58
intron_variant                               8
non_coding_transcript_exon_variant           6
protein_protein_contact  